In [1]:
m_roster ={
    2021:["/teams/tf/CA_college_m_UC_San_Diego.html?config_hnd=214",
          "/teams/tf/CA_college_m_Cal_St_Bakersfield.html?config_hnd=214"],
    2022:["/teams/tf/CA_college_m_UC_San_Diego.html?config_hnd=254",
          "/teams/tf/CA_college_m_Cal_St_Bakersfield.html?config_hnd=254"],
    2023:["/teams/tf/CA_college_m_UC_San_Diego.html?config_hnd=290"],
    2024:["/teams/tf/CA_college_m_UC_San_Diego.html?config_hnd=333"],
    2025:["/teams/tf/CA_college_m_UC_San_Diego.html?config_hnd=380"],
    2026:["/teams/tf/CA_college_m_UC_San_Diego.html?config_hnd=432"]} 
w_roster = {
    2021:["/teams/tf/CA_college_f_UC_San_Diego.html?config_hnd=214",
          "/teams/tf/CA_college_f_Cal_St_Bakersfield.html?config_hnd=214"],
    2022:["/teams/tf/CA_college_f_UC_San_Diego.html?config_hnd=254",
          "/teams/tf/CA_college_f_Cal_St_Bakersfield.html?config_hnd=254"],
    2023:["/teams/tf/CA_college_f_UC_San_Diego.html?config_hnd=290"],
    2024:["/teams/tf/CA_college_f_UC_San_Diego.html?config_hnd=333"],
    2025:["/teams/tf/CA_college_f_UC_San_Diego.html?config_hnd=380"],
    2026:["/teams/tf/CA_college_f_UC_San_Diego.html?config_hnd=432"] 
}

In [2]:
years_df = {
    2021: 214,
    2022: 254,
    2023: 290,
    2024: 333,
    2025: 380,
    2026: 432
}

gender = ["m", "f"]

schools = [
    "UC_San_Diego",
    "Cal_St_Bakersfield",
    "Cal_St_Northridge",
    "Cal_St_Fullerton",
    "Long_Beach_St",
    "UC_Davis",
    "UC_Irvine",
    "UC_Riverside",
    "UC_Santa_Barbara",
    "Cal_Poly"
    
]

def generate_roster_urls(years_df, gender, schools):
    urls = []

    for year, config_hnd in years_df.items():
        for g in gender:
            for school in schools:

                url = (
                    f"/teams/tf/CA_college_{g}_{school}.html"
                    f"?config_hnd={config_hnd}"
                )

                urls.append({
                    "year": year,
                    "gender": g,
                    "school": school,
                    "url": url
                })

    return urls

roster_urls = generate_roster_urls(years_df, gender, schools)

In [3]:
roster_urls

[{'year': 2021,
  'gender': 'm',
  'school': 'UC_San_Diego',
  'url': '/teams/tf/CA_college_m_UC_San_Diego.html?config_hnd=214'},
 {'year': 2021,
  'gender': 'm',
  'school': 'Cal_St_Bakersfield',
  'url': '/teams/tf/CA_college_m_Cal_St_Bakersfield.html?config_hnd=214'},
 {'year': 2021,
  'gender': 'm',
  'school': 'Cal_St_Northridge',
  'url': '/teams/tf/CA_college_m_Cal_St_Northridge.html?config_hnd=214'},
 {'year': 2021,
  'gender': 'm',
  'school': 'Cal_St_Fullerton',
  'url': '/teams/tf/CA_college_m_Cal_St_Fullerton.html?config_hnd=214'},
 {'year': 2021,
  'gender': 'm',
  'school': 'Long_Beach_St',
  'url': '/teams/tf/CA_college_m_Long_Beach_St.html?config_hnd=214'},
 {'year': 2021,
  'gender': 'm',
  'school': 'UC_Davis',
  'url': '/teams/tf/CA_college_m_UC_Davis.html?config_hnd=214'},
 {'year': 2021,
  'gender': 'm',
  'school': 'UC_Irvine',
  'url': '/teams/tf/CA_college_m_UC_Irvine.html?config_hnd=214'},
 {'year': 2021,
  'gender': 'm',
  'school': 'UC_Riverside',
  'url': '/

In [4]:
years_df = {
    2021: 214,
    2022: 254,
    2023: 290,
    2024: 333,
    2025: 380,
    2026: 432
}

hawaii_urls = []

for year, config_hnd in years_df.items():
    url = f"/teams/tf/HI_college_f_Hawaii.html?config_hnd={config_hnd}"

    hawaii_urls.append({
        "year": year,
        "gender": "f",
        "school": "Hawaii",
        "url": url
    })
roster_urls.extend(hawaii_urls)

## Add API Key if you want to scrape

In [7]:
import pandas as pd
from pathlib import Path

PROGRESS_FILE = Path("../data/processed/completed_roster_urls.csv")

# Cal Poly is the only school not previously collected
MISSING_SCHOOLS = {"Cal_Poly"}

# Load any existing completed URL history
if PROGRESS_FILE.exists():
    completed_urls = set(
        pd.read_csv(PROGRESS_FILE)["url"].dropna()
    )
else:
    completed_urls = set()

# Mark previously scraped schools as completed
for info in roster_urls:
    if info["school"] not in MISSING_SCHOOLS:
        completed_urls.add(info["url"])

# Save completed URL history
pd.DataFrame({
    "url": sorted(completed_urls)
}).to_csv(PROGRESS_FILE, index=False)

print("Completed URLs recorded:", len(completed_urls))

Completed URLs recorded: 114


In [9]:
import requests
import pandas as pd
import time
import os
from pathlib import Path

api_url = (
    "https://api.parse.bot/scraper/"
    "6e80f113-decc-4a5f-960a-70b2f1d469bc/"
    "get_team_roster"
)

ROSTER_FILE = Path("../data/processed/bigwest_roster.csv")
PROGRESS_FILE = Path("../data/processed/completed_roster_urls.csv")

RUN_SCRAPER = True      # Change to True when ready
MAX_REQUESTS = 20        # Maximum API calls per run
REQUEST_DELAY = 3.5      # Delay between requests (seconds)


# ==========================================
# HELPER FUNCTION
# ==========================================

def format_name(name):
    if "," in name:
        last, first = name.split(",", 1)
        return f"{first.strip()} {last.strip()}"
    return name.strip()


# ==========================================
# LOAD EXISTING ROSTER DATA
# ==========================================

ROSTER_FILE.parent.mkdir(parents=True, exist_ok=True)

if ROSTER_FILE.exists():

    roster_df = pd.read_csv(
        ROSTER_FILE,
        dtype={"athlete_id": "string"}
    )

    print(f"Loaded {len(roster_df)} existing athlete records.")

else:

    roster_df = pd.DataFrame(
        columns=["athlete_id", "year", "school"]
    )

    print("No existing roster file found.")


# ==========================================
# LOAD COMPLETED URL HISTORY
# ==========================================

if PROGRESS_FILE.exists():

    completed_urls = set(
        pd.read_csv(PROGRESS_FILE)["url"]
        .dropna()
        .astype(str)
    )

    print(f"Loaded {len(completed_urls)} completed URLs.")

else:

    completed_urls = set()

    print("No completed URL history found.")
    print("Previously scraped URLs must be verified manually.")


# ==========================================
# FIND MISSING ROSTERS
# ==========================================

# Remove duplicate URLs
unique_urls = {
    info["url"]: info
    for info in roster_urls
}

# Identify URLs that have not been completed
remaining_rosters = [
    info
    for url, info in unique_urls.items()
    if url not in completed_urls
]


# ==========================================
# PREVIEW (NO API REQUESTS)
# ==========================================

print("\n" + "=" * 50)
print("ROSTER SCRAPING PREVIEW")
print("=" * 50)

print(f"Total roster URLs: {len(unique_urls)}")
print(
    "Previously completed:",
    sum(url in completed_urls for url in unique_urls)
)
print(f"Remaining URLs: {len(remaining_rosters)}")
print(f"Maximum requests this run: {MAX_REQUESTS}")

print("\nROSTERS STILL NEEDING COLLECTION:\n")

for info in remaining_rosters:
    print(
        f"{info['year']} | "
        f"{info['gender']} | "
        f"{info['school']}"
    )


# ==========================================
# SAFETY CHECK
# ==========================================

if not RUN_SCRAPER:

    print("\nPREVIEW MODE")
    print("No API requests were made.")
    print("Set RUN_SCRAPER = True to start collection.")

elif len(remaining_rosters) == 0:

    print("\nAll roster URLs have been completed.")

else:

    # Only process up to MAX_REQUESTS
    rosters_to_scrape = remaining_rosters[:MAX_REQUESTS]

    print("\n" + "=" * 50)
    print("STARTING ROSTER COLLECTION")
    print("=" * 50)

    print(f"Requests planned: {len(rosters_to_scrape)}")

    headers = {
        "X-API-Key": "pmx_e2173f90c664df059780119f36d518e4"
    }

    requests_made = 0
    successful_requests = 0
    failed_requests = []

    try:

        for i, info in enumerate(rosters_to_scrape, start=1):

            print(
                f"\n[{i}/{len(rosters_to_scrape)}] "
                f"{info['school']} | "
                f"{info['year']} | "
                f"{info['gender']}"
            )

            try:

                requests_made += 1

                response = requests.get(
                    api_url,
                    headers=headers,
                    params={"team_url": info["url"]},
                    timeout=120
                )

                # Stop if rate limited
                if response.status_code == 429:

                    print("Rate limit reached.")

                    print(
                        "Retry after:",
                        response.headers.get("Retry-After")
                    )

                    print(
                        "Daily remaining:",
                        response.headers.get(
                            "x-ratelimit-daily-remaining"
                        )
                    )

                    break

                response.raise_for_status()

                result = response.json()

                if result.get("status") != "success":
                    print("API returned unsuccessful status.")
                    failed_requests.append(info)
                    continue

                data = result["data"]

                # ==========================================
                # EXTRACT ATHLETES
                # ==========================================

                rows = []

                for athlete in data.get("roster", []):

                    rows.append({
                        "athlete_id": str(athlete["athlete_id"]),
                        "year": info["year"],
                        "school": data.get(
                            "team_name",
                            info["school"]
                        )
                    })

                # ==========================================
                # SAVE ROSTER DATA
                # ==========================================

                if rows:

                    roster_df = pd.concat(
                        [
                            roster_df,
                            pd.DataFrame(rows)
                        ],
                        ignore_index=True
                    )

                    roster_df = roster_df.drop_duplicates(
                        subset=["athlete_id", "year", "school"]
                    )

                roster_df.to_csv(
                    ROSTER_FILE,
                    index=False
                )

                # ==========================================
                # SAVE COMPLETED URL
                # ==========================================

                completed_urls.add(info["url"])

                pd.DataFrame({
                    "url": sorted(completed_urls)
                }).to_csv(
                    PROGRESS_FILE,
                    index=False
                )

                successful_requests += 1

                print(
                    f"Success! Collected {len(rows)} athletes."
                )

                print(
                    f"Progress saved: "
                    f"{len(completed_urls)} completed URLs."
                )

            except (
                requests.RequestException,
                ValueError,
                KeyError,
                TypeError
            ) as e:

                print(f"Failed: {e}")
                failed_requests.append(info)

            # Respect rate limit
            time.sleep(REQUEST_DELAY)

    except KeyboardInterrupt:

        print("\nCollection interrupted.")
        print("All previously saved progress is preserved.")


    # ==========================================
    # FINAL SUMMARY
    # ==========================================

    print("\n" + "=" * 50)
    print("COLLECTION SUMMARY")
    print("=" * 50)

    print("API requests made:", requests_made)
    print("Successful requests:", successful_requests)
    print("Failed requests:", len(failed_requests))
    print("Total athlete records:", len(roster_df))

    remaining_after = [
        info
        for url, info in unique_urls.items()
        if url not in completed_urls
    ]

    print("Remaining URLs:", len(remaining_after))

Loaded 4709 existing athlete records.
Loaded 114 completed URLs.

ROSTER SCRAPING PREVIEW
Total roster URLs: 126
Previously completed: 114
Remaining URLs: 12
Maximum requests this run: 20

ROSTERS STILL NEEDING COLLECTION:

2021 | m | Cal_Poly
2021 | f | Cal_Poly
2022 | m | Cal_Poly
2022 | f | Cal_Poly
2023 | m | Cal_Poly
2023 | f | Cal_Poly
2024 | m | Cal_Poly
2024 | f | Cal_Poly
2025 | m | Cal_Poly
2025 | f | Cal_Poly
2026 | m | Cal_Poly
2026 | f | Cal_Poly

STARTING ROSTER COLLECTION
Requests planned: 12

[1/12] Cal_Poly | 2021 | m
Success! Collected 68 athletes.
Progress saved: 115 completed URLs.

[2/12] Cal_Poly | 2021 | f
Success! Collected 48 athletes.
Progress saved: 116 completed URLs.

[3/12] Cal_Poly | 2022 | m
Success! Collected 56 athletes.
Progress saved: 117 completed URLs.

[4/12] Cal_Poly | 2022 | f
Success! Collected 51 athletes.
Progress saved: 118 completed URLs.

[5/12] Cal_Poly | 2023 | m
Success! Collected 48 athletes.
Progress saved: 119 completed URLs.

[6/12]

<StringArray>
[     'UC SAN DIEGO',   'CSU BAKERSFIELD',              'CSUN',
 'CAL ST. FULLERTON',    'LONG BEACH ST.',          'UC DAVIS',
         'UC IRVINE',      'UC RIVERSIDE',  'UC SANTA BARBARA',
            'HAWAII',          'CAL POLY']
Length: 11, dtype: str